# ForecastEx

## Web REST API

In [ ]:
import requests, json, os
from pprint import pprint

## Get the current markets

In [ ]:
from get_forecastex_markets import get_forecastex_markets

In [ ]:
fn_market = 'forecastex_markets.json'
if os.path.exists(fn_market):
    with open(fn_market, 'r') as f:
        markets = json.load(f)
else:
    response = requests.get(
        'https://localhost:5000/v1/api/trsrv/event/category-tree',
        verify=False  # skip SSL verification for local gateway
    )
    markets=get_forecastex_markets(response.json())
    with open('forecastex_markets.json', 'w') as f:
        json.dump(markets, f, indent=4)

## Scrape the markets site

In [ ]:
import requests
from collections import defaultdict
market_url = f"https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/contracts"
pricing_url = f"https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/binaryoptions"

In [ ]:
# -- Step 1: Get all contracts under the market
def fetch_all_contracts(market_conid):
    params = {
        "showrestricted": "false",
        "market": market_conid
    }
    r = requests.get(market_url, params=params)
    r.raise_for_status()
    contracts = r.json()["contracts"]
    return contracts

In [ ]:
market_conid = "796056051"

In [ ]:
contracts = fetch_all_contracts(market_conid)

In [ ]:
contracts[0]

In [ ]:
# -- Step 2: Group contracts by candidate with YES and NO
def extract_candidate_conids(contract_data):
    candidates = defaultdict(dict)
    for c in contract_data:
        name = c.get("strikeLabel")
        direction = "YES" if c.get("putOrCall") == "C" else "NO"
        candidates[name][direction] = {
            "conid": c["conid"],
            "description": c["shortDescription"]
        }
    return candidates

In [ ]:
candidates = extract_candidate_conids(contracts)

In [ ]:
candidates

In [ ]:
conids = [side["conid"] for c in candidates.values() for side in c.values()]

In [ ]:
conids

## Getting somewhere

In [ ]:
from get_candidate_probability import get_candidate_probability

In [ ]:
result = get_candidate_probability(796056520)  # Mamdani YES conid
print(f"✅ Mamdani latest probability: {result['probability_pct']:.1f}% (as of {result['timestamp']})")

In [ ]:
conid = 796056520

In [ ]:
period="1week"

In [ ]:
    url = "https://forecasttrader.interactivebrokers.com/tws.proxy/public/hmds/forecastContract"
    params = {
        "conid": conid,
        "period": period,
        "exchange": "FORECASTX",
        "secType": "OPT"
    }

    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()


In [ ]:
[x for x in data]

In [ ]:
result

## Final version get market data

In [ ]:
market = {'conid': 796056051, 'name': 'General Election for New York City Mayor', 'symbol': 'MNYCG'}

### Discover All YES/NO Candidate Subcontracts

In [ ]:
import requests

# Market container for NYC 2025
market_conid = str(market["conid"])
contracts_url = "https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/contracts"
contracts_params = {"showrestricted":"false", "market":market_conid}

contracts_resp = requests.get(contracts_url, params=contracts_params)
contracts_resp.raise_for_status()
contracts_raw = contracts_resp.json()["contracts"]

# Organize by candidate and side
from collections import defaultdict
candidates = defaultdict(dict)
for c in contracts_raw:
    name = c["strikeLabel"]
    side = "YES" if c["putOrCall"] == "C" else "NO"
    candidates[name][side] = c["conid"]

print("CANDIDATES & CONIDs:")
for name, sides in candidates.items():
    print(f"{name:10} YES: {sides['YES']} NO: {sides['NO']}")


###  Get LIVE PRICING for All Candidates

In [ ]:
# Build list of all YES and NO conids
conid_list = []
for sides in candidates.values():
    conid_list.extend([sides["YES"], sides["NO"]])

pricing_url = "https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/binaryoptions"
pricing_params = {"conids": ",".join(str(cid) for cid in conid_list)}

pricing_resp = requests.get(pricing_url, params=pricing_params)
pricing_resp.raise_for_status()
live_data = pricing_resp.json()

# Build a lookup dict by conid
live_lookup = {d["conid"]: d for d in live_data}

### Get LATEST HISTORICAL "LINE CHART" VALUE for Each YES Contract

In [ ]:
def get_latest_probability(conid):
    """Returns the latest YES probability from forecastContract chart, or None if unavailable."""
    url = "https://forecasttrader.interactivebrokers.com/tws.proxy/public/hmds/forecastContract"
    params = {
        "conid": conid,
        "period": "1week",
        "exchange": "FORECASTX",
        "secType": "OPT"
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()
    avg = data.get("avg")
    if not avg:
        return None
    return round(avg[-1] * 100, 2)

### Print the Full Per-Candidate Table (Current, Bid/Ask, Historical Line)

In [ ]:
print("\nNYC Mayor Election Market — ForecastEx Live Snapshot")
print(f"{'Candidate':10s} {'YES Line%':>9s}")

for name, sides in candidates.items():
    yes = live_lookup.get(sides["YES"], {})
    no  = live_lookup.get(sides["NO"], {})
    # Most recent line chart/YES-probability
    yes_prob = get_latest_probability(sides["YES"])
    line_str = f"{yes_prob:>9.2f}" if yes_prob is not None else "    —    "
    print(f"{name:10s}  "
          f"{line_str}")

In [ ]:
import requests

# The target endpoint
url = "https://forecasttrader.interactivebrokers.com/tws.proxy/public/hmds/forecastContract"

# Query parameters
params = {
    "conid": "796056520",
    "period": "1week",
    "exchange": "FORECASTX",
    "secType": "OPT"
}

# HTTP headers, skipping HTTP/2 pseudo-headers (':authority', ':method', ':path', ':scheme')
headers = {
    "accept": "*/*",
    "accept-encoding": "gzip, deflate, br, zstd",
    "accept-language": "en-US,en;q=0.9",
    "cache-control": "no-cache",
    "content-type": "application/json; charset=utf-8",
    "dnt": "1",
    "pragma": "no-cache",
    "priority": "u=1, i",
    "referer": "https://forecasttrader.interactivebrokers.com/eventtrader/",
    "sec-ch-ua": "\"Google Chrome\";v=\"137\", \"Chromium\";v=\"137\", \"Not/A)Brand\";v=\"24\"",
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": "\"Linux\"",
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-origin",
    "user-agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
    "x-ccp-session-id": "undefined",
    "x-client-label": "IB",
    "x-embedded-in": "web",
    "x-request-id": "45",
    "x-service": "AM.LOGIN",
    "x-session-id": "334fa90c-aca9-42ef-8502-72ab677f34b7",
    "x-wa-version": "3aadda4e,Wed, 9 Jul 2025 21:57:03 +0000/2025-07-09T22:02:47.723Z"
}

# Execute the GET request
response = requests.get(url, params=params, headers=headers)

# Print the response
print(response.status_code)

In [ ]:
print(response.json())  # Use .text if the response is not JSON

## Category Tree (no need for local login)

In [ ]:
import requests

url = "https://forecasttrader.interactivebrokers.com/tws.proxy/public/forecasttrader/category/tree"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
}

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    json_data = response.json()
    print("JSON response:")
    print(json_data)
except Exception as e:
    print("Error during request:", e)


## Open Interest (must run chrome in debug mode)

https://www.perplexity.ai/search/how-do-i-do-this-request-in-py-LKWOZo0lRNegn9r7ALA1dw

In [1]:
import websocket
import threading
import json
import asyncio
import requests
import websockets

# --- Step 1: Auto-fetch cookies from Chrome's DevTools session ---
def get_websocket_debugger_url(domain="forecasttrader.interactivebrokers.com", port=9222):
    tabs = requests.get(f"http://localhost:{port}/json").json()
    for tab in tabs:
        if domain in tab.get("url", ""):
            return tab["webSocketDebuggerUrl"]
    raise Exception("ForecastTrader tab not found.")

async def get_cookie_header_from_chrome_devtools():
    ws_url = get_websocket_debugger_url()
    async with websockets.connect(ws_url) as ws:
        await ws.send(json.dumps({"id": 1, "method": "Network.enable"}))
        await ws.recv()  # Discard ack

        await ws.send(json.dumps({"id": 2, "method": "Network.getAllCookies"}))

        while True:
            response = await ws.recv()
            message = json.loads(response)
            if message.get("id") == 2:
                cookies = message["result"]["cookies"]
                relevant = [
                    f"{c['name']}={c['value']}"
                    for c in cookies
                    if "interactivebrokers.com" in c["domain"]
                ]
                return "; ".join(relevant)

# --- Step 2: Get OI Mapping via WebSocket ---
def get_OI_for_conids(conids, cookie_header):
    WS_URL = "wss://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/ws"
    FIELDS = ["7638"]
    oi_results = {}
    received_conids = set()
    lock = threading.Lock()
    done_event = threading.Event()

    # Prepare headers
    headers = [
        "Origin: https://forecasttrader.interactivebrokers.com",
        "Referer: https://forecasttrader.interactivebrokers.com/en/home.php",
        "User-Agent: Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
        f"Cookie: {cookie_header}"
    ]

    def on_open(ws):
        for conid in conids:
            msg = f'smd+{conid}+{json.dumps({"fields": FIELDS, "backout": True})}'
            ws.send(msg)

    def on_message(ws, message):
        try:
            data = json.loads(message)
            if "7638" in data and "conid" in data:
                conid = str(data["conid"])
                oi = data["7638"]
                with lock:
                    if conid not in received_conids:
                        oi_results[conid] = oi
                        received_conids.add(conid)
                    if len(received_conids) >= len(conids):
                        done_event.set()
                        ws.close()
        except Exception:
            pass

    def on_error(ws, error):
        print("❌ WebSocket error:", error)
        done_event.set()

    def on_close(ws, code, msg):
        done_event.set()

    ws_app = websocket.WebSocketApp(
        WS_URL,
        header=headers,
        on_open=on_open,
        on_message=on_message,
        on_error=on_error,
        on_close=on_close
    )

    thread = threading.Thread(target=ws_app.run_forever)
    thread.daemon = True
    thread.start()

    done_event.wait(timeout=20)
    if ws_app.keep_running:
        ws_app.close()
    return oi_results

# --- Step 3: Async wrapper to run it all ---
async def run_get_OI(conids):
    cookie_header = await get_cookie_header_from_chrome_devtools()
    result = get_OI_for_conids(conids, cookie_header)
    print("\n✅ Final Result:\n", result)
    return result

# --- Example usage ---
# Run this in an async environment like Jupyter or an asyncio-compatible main
await run_get_OI(["796056520", "796056525"])



✅ Final Result:
 {'796056520': '1.16M', '796056525': '1.16M'}


{'796056520': '1.16M', '796056525': '1.16M'}